https://tianchi.aliyun.com/notebook/144449

## Task1 基础理解Baseline

赛题理解是切入一道赛题的基础，会影响后续特征工程和模型构建等各种工作，也影响着后续发展工作的方向，正确了解赛题背后的思想以及赛题业务逻辑的清晰，有利于花费更少时间构建更为有效的特征模型。

根据赛题简介，我们首先要明确我们此次比赛的目标： 根据用户历史浏览点击新闻的数据信息预测用户最后一次点击的新闻文章。从这个目标上看， 会发现此次比赛和我们之前遇到的普通的结构化比赛不太一样， 主要有两点：

- 首先是目标上， 要预测最后一次点击的新闻文章，也就是我们给用户推荐的是新闻文章， 并不是像之前那种预测一个数或者预测数据哪一类那样的问题

- 数据上， 通过给出的数据我们会发现， 这种数据也不是我们之前遇到的那种特征+标签的数据，而是基于了真实的业务场景， 拿到的用户的点击日志


所以拿到这个题目，我们的思考方向就是结合我们的目标，把该预测问题转成一个监督学习的问题(特征+标签)，然后我们才能进行ML，DL等建模预测。那么我们自然而然的就应该在心里会有这么几个问题：如何转成一个监督学习问题呢？ 转成一个什么样的监督学习问题呢？ 我们能利用的特征又有哪些呢？ 又有哪些模型可以尝试呢？ 此次面对数万级别的文章推荐，我们又有哪些策略呢？

当然这些问题不会在我们刚看到赛题之后就一下出来答案， 但是只要有了问题之后， 我们就能想办法解决问题了， 比如上面的第二个问题，转成一个什么样的监督学习问题？ 由于我们是预测用户最后一次点击的新闻文章，从36万篇文章中预测某一篇的话我们首先可能会想到这可能是一个多分类的问题(36万类里面选1)， 但是如此庞大的分类问题， 我们做起来可能比较困难， 那么能不能转化一下？ 既然是要预测最后一次点击的文章， 那么如果我们能预测出某个用户最后一次对于某一篇文章会进行点击的概率， 是不是就间接性的解决了这个问题呢？概率最大的那篇文章不就是用户最后一次可能点击的新闻文章吗？ 这样就把原问题变成了一个点击率预测的问题(用户, 文章) --> 点击的概率(软分类)， 而这个问题， 就是我们所熟悉的监督学习领域分类问题了， 这样我们后面建模的时候， 对于模型的选择就基本上有大致方向了，比如最简单的逻辑回归模型。

这样， 我们对于该赛题的解决方案应该有了一个大致的解决思路，要先转成一个分类问题来做， 而分类的标签就是用户是否会点击某篇文章，分类问题的特征中会有用户和文章，我们要训练一个分类模型， 对某用户最后一次点击某篇文章的概率进行预测。 那么又会有几个问题：如何转成监督学习问题？ 训练集和测试集怎么制作？ 我们又能利用哪些特征？ 我们又可以尝试哪些模型？ 面对36万篇文章， 20多万用户的推荐， 我们又有哪些策略来缩减问题的规模？如何进行最后的预测？

## 整理总结

- 赛题理解究竟是在理解什么?

    >理解赛题：从直观上对问题进行梳理， 分析问题的目标，到底要让做什么事情, 这个非常重要

    >理解数据：对赛题数据有一个初步了解，知道和任务相关的数据字段和数据字段的类型， 数据之间的内在关联等，大体梳理一下哪些数据会对我们解决问题非常有用，方便后面我们的数据分析和特征工程。

    >理解评估指标：评估指标是检验我们提出的方法，我们给出结果好坏的标准，只有正确的理解了评估指标，我们才能进行更好的训练模型，更好的进行预测。此外，很多情况下，线上验证是有一定的时间和次数限制的，所以在比赛中构建一个合理的本地的验证集和验证的评价指标是很关键的步骤，能有效的节省很多时间。 不同的指标对于同样的预测结果是具有误差敏感的差异性的所以不同的评价指标会影响后续一些预测的侧重点。

- 有了赛题理解之后，我们该做什么？

    >在对于赛题有了一定的了解后，分析清楚了问题的类型性质和对于数据理解 的这一基础上，我们可以梳理一个解决赛题的一个大题思路和框架

    >我们至少要有一些相应的理解分析，比如这题的难点可能在哪里，关键点可能在哪里，哪些地方可以挖掘更好的特征.

    >用什么样得线下验证方式更为稳定，出现了过拟合或者其他问题，估摸可以用什么方法去解决这些问题

    这时是在一个宏观的大体下分析的，有助于摸清整个题的思路脉络，以及后续的分析方向

## 导包

In [1]:
# import packages
import time, math, os, sys, json
from pathlib import Path
from tqdm import tqdm
import gc
import pickle
import random
from datetime import datetime
from operator import itemgetter
import numpy as np
import pandas as pd
import warnings
from collections import defaultdict
import collections
warnings.filterwarnings('ignore')

In [2]:
# 支持从项目根目录或 Task-1 启动 notebook
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'metric').is_dir() and (p / 'data').is_dir())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from metric.data import prepare_evaluation
from metric.evaluation import evaluate_csv
data_path = str(project_root / 'data') + os.sep
results_path = project_root / 'Task-1' / 'results'

## df节省内存函数

In [3]:
# 节约内存的一个标配函数
def reduce_mem(df):
    starttime = time.time()
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.isnull(c_min) or pd.isnull(c_max):
                continue
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    print('-- Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction),time spend:{:2.2f} min'.format(end_mem,
                                                                                                           100*(start_mem-end_mem)/start_mem,
                                                                                                           (time.time()-starttime)/60))
    return df

## 读取采样或全量数据

In [5]:
# debug模式：从训练集中划出一部分数据来调试代码
def get_all_click_sample(data_path, sample_nums=10000):
    """
        训练集中采样一部分数据调试
        data_path: 原数据的存储路径
        sample_nums: 采样数目（这里由于机器的内存限制，可以采样用户做）
    """
    all_click = pd.read_csv(data_path + 'train_click_log.csv')
    all_user_ids = all_click.user_id.unique()

    sample_user_ids = np.random.choice(all_user_ids, size=sample_nums, replace=False) 
    all_click = all_click[all_click['user_id'].isin(sample_user_ids)]
    
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click

# 该工具函数只选择原始数据来源；离线划分由后面的 metric 模块完成
def get_all_click_df(data_path='./data_raw/', include_test=False):
    if not include_test:
        all_click = pd.read_csv(data_path + 'train_click_log.csv')
    else:
        trn_click = pd.read_csv(data_path + 'train_click_log.csv')
        tst_click = pd.read_csv(data_path + 'testA_click_log.csv')

        all_click = pd.concat([trn_click, tst_click], ignore_index=True)
    
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click

In [6]:
# True：固定 train 留一评估；False：用全部可见点击生成线上提交格式 CSV

offline = True
mode = 'offline_eval' if offline else 'online'
save_path = str(results_path / mode) + os.sep
cache_path = results_path.parent / 'cache' / mode
Path(save_path).mkdir(parents=True, exist_ok=True)
cache_path.mkdir(parents=True, exist_ok=True)
if offline:
    click_columns = ['user_id', 'click_article_id', 'click_timestamp']
    train_click_df = pd.read_csv(data_path + 'train_click_log.csv', usecols=click_columns)
    # 公共划分供所有 Task 复用；首次保存三份 CSV，后续直接读取，不重新抽样
    eval_cache_path = project_root / 'cache' / 'offline_eval'
    fit_clicks, query_clicks, eval_targets = prepare_evaluation(
        train_click_df, n_users=50_000, random_state=42, cache_dir=eval_cache_path,
    )
    # 验证用户只移除末次点击，其余历史仍参与相似度和热门统计
    all_click_df, query_click_df = fit_clicks, query_clicks
    targets_path = eval_cache_path / 'targets.csv'
else:
    all_click_df = get_all_click_df(data_path, include_test=True)
    query_click_df = pd.read_csv(data_path + 'testA_click_log.csv')

## 获取 用户-文章-点击时间 字典

In [7]:
# 根据点击时间获取用户的点击文章序列   {user1: [(item1, time1), (item2, time2)..]...}
def get_user_item_time(click_df):
    
    click_df = click_df.sort_values('click_timestamp')
    
    def make_item_time_pair(df):
        return list(zip(df['click_article_id'], df['click_timestamp']))
    
    user_item_time_df = click_df.groupby('user_id')[['click_article_id', 'click_timestamp']].apply(lambda x: make_item_time_pair(x))\
                                                            .reset_index().rename(columns={0: 'item_time_list'})
    user_item_time_dict = dict(zip(user_item_time_df['user_id'], user_item_time_df['item_time_list']))
    
    return user_item_time_dict

## 获取点击最多的top-k个文章

In [8]:
# 获取近期点击最多的文章
def get_item_topk_click(click_df, k):
    topk_click = click_df['click_article_id'].value_counts().index[:k]
    return topk_click

## itemcf的物品相似度计算

In [9]:
def itemcf_sim(df):
    """
        文章与文章之间的相似性矩阵计算
        :param df: 数据表
        :item_created_time_dict:  文章创建时间的字典
        return : 文章与文章的相似性矩阵
        思路: 基于物品的协同过滤(详细请参考上一期推荐系统基础的组队学习)， 在多路召回部分会加上关联规则的召回策略
    """
    
    user_item_time_dict = get_user_item_time(df)
    
    # 计算物品相似度
    i2i_sim = {}
    item_cnt = defaultdict(int)
    for user, item_time_list in tqdm(user_item_time_dict.items()):
        # 在基于商品的协同过滤优化的时候可以考虑时间因素
        for i, i_click_time in item_time_list:
            item_cnt[i] += 1
            i2i_sim.setdefault(i, {})
            for j, j_click_time in item_time_list:
                if(i == j):
                    continue
                i2i_sim[i].setdefault(j, 0)
                
                i2i_sim[i][j] += 1 / math.log(len(item_time_list) + 1)
                
    i2i_sim_ = i2i_sim.copy()
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j] = wij / math.sqrt(item_cnt[i] * item_cnt[j])
    
    # 将得到的相似性矩阵保存到本地
    with open(cache_path / 'similarity.pkl', 'wb') as stream:
        pickle.dump(i2i_sim_, stream)
    
    return i2i_sim_

In [ ]:
i2i_sim = itemcf_sim(all_click_df)

## itemcf 的文章推荐

In [10]:
# 基于商品的召回i2i
def item_based_recommend(user_id, user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num, item_topk_click):
    """
        基于文章协同过滤的召回
        :param user_id: 用户id
        :param user_item_time_dict: 字典, 根据点击时间获取用户的点击文章序列   {user1: [(item1, time1), (item2, time2)..]...}
        :param i2i_sim: 字典，文章相似性矩阵
        :param sim_item_topk: 整数， 选择与当前文章最相似的前k篇文章
        :param recall_item_num: 整数， 最后的召回文章数量
        :param item_topk_click: 列表，点击次数最多的文章列表，用户召回补全        
        return: 召回的文章列表 {item1:score1, item2: score2...}
        注意: 基于物品的协同过滤(详细请参考上一期推荐系统基础的组队学习)， 在多路召回部分会加上关联规则的召回策略
    """
    
    # 获取用户历史交互的文章
    user_hist_items = user_item_time_dict[user_id]
    user_hist_items_ = {user_id for user_id, _ in user_hist_items}
    
    item_rank = {}
    for loc, (i, click_time) in enumerate(user_hist_items):
        for j, wij in sorted(i2i_sim.get(i, {}).items(), key=lambda x: (-x[1], x[0]))[:sim_item_topk]:
            if j in user_hist_items_:
                continue
                
            item_rank.setdefault(j, 0)
            item_rank[j] +=  wij
    
    # 候选不足时用热门文章补全，保留已有分数并过滤已读文章
    if len(item_rank) < recall_item_num:
        for i, item in enumerate(item_topk_click):
            if item in item_rank or item in user_hist_items_:
                continue
            item_rank[item] = - i - 100 # 随便给个负数就行
            if len(item_rank) == recall_item_num:
                break
    
    item_rank = sorted(item_rank.items(), key=lambda x: (-x[1], x[0]))[:recall_item_num]
        
    return item_rank

## 给每个用户根据物品的协同过滤推荐文章

In [11]:
# 定义
user_recall_items_dict = collections.defaultdict(dict)

# 获取 用户 - 文章 - 点击时间的字典
user_item_time_dict = get_user_item_time(query_click_df)

# 去取文章相似度
with open(cache_path / 'similarity.pkl', 'rb') as stream:
    i2i_sim = pickle.load(stream)

# 相似文章的数量
sim_item_topk = 10

# 召回文章数量
recall_item_num = 50  

# 用户热度补全
item_topk_click = get_item_topk_click(all_click_df, k=50)

# 邻居只需预排序一次，避免每个用户反复排序完整相似度字典
i2i_topk = {item: dict(sorted(neighbors.items(), key=lambda x: (-x[1], x[0]))[:sim_item_topk])
           for item, neighbors in i2i_sim.items()}
for user in tqdm(query_click_df['user_id'].unique()):
    user_recall_items_dict[user] = item_based_recommend(user, user_item_time_dict, i2i_topk, 
                                                        sim_item_topk, recall_item_num, item_topk_click)

100%|██████████| 50000/50000 [00:01<00:00, 25755.09it/s]


## 召回字典转换成df

In [12]:
# 将字典的形式转换成df
user_item_score_list = []

for user, items in tqdm(user_recall_items_dict.items()):
    for item, score in items:
        user_item_score_list.append([user, item, score])

recall_df = pd.DataFrame(user_item_score_list, columns=['user_id', 'click_article_id', 'pred_score'])

100%|██████████| 50000/50000 [00:01<00:00, 29004.07it/s]


## 保存预测 CSV 并进行本地离线评估

In [13]:
# 离线允许不足 K 篇和零候选，尾部留空；这些用户仍进入评估分母
def submit(recall_df, topk=5, user_ids=None, require_complete=True):
    ranked = recall_df.sort_values(['user_id', 'pred_score', 'click_article_id'],
                                 ascending=[True, False, True]).copy()
    if ranked.duplicated(['user_id', 'click_article_id']).any():
        raise ValueError('同一用户的候选文章存在重复。')
    ranked['rank'] = ranked.groupby('user_id').cumcount() + 1
    users = pd.Index(pd.unique(user_ids if user_ids is not None else ranked['user_id']), name='user_id')
    counts = ranked.groupby('user_id').size().reindex(users, fill_value=0)
    if require_complete and (len(users) == 0 or counts.lt(topk).any()):
        raise ValueError(f'生成线上提交要求每个目标用户至少有 {topk} 篇候选文章。')
    output = ranked.loc[ranked['rank'] <= topk].pivot(index='user_id', columns='rank', values='click_article_id')
    output = output.reindex(index=users, columns=range(1, topk + 1)).astype('Int64')
    output.columns = [f'article_{rank}' for rank in range(1, topk + 1)]
    save_name = Path(save_path) / f'top-{topk}.csv'
    output.reset_index().to_csv(save_name, index=False)
    
    return save_name

In [14]:
# 只导出本次 query 中的用户；离线为固定的 50,000 个验证用户
query_users = query_click_df['user_id'].unique()
query_recall = recall_df[recall_df['user_id'].isin(query_users)]
if offline:
    prediction_path = submit(query_recall, topk=recall_item_num,
                             user_ids=query_users, require_complete=False)
    metrics = evaluate_csv(prediction_path, targets_path, train_users=all_click_df['user_id'].nunique(),
                           output_path=Path(save_path) / 'metrics.json')
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    # 仅生成本地 CSV，不会自动上传
    prediction_path = submit(query_recall, topk=5, user_ids=query_users)
    print(f'提交文件：{prediction_path}')

{
  "train_users": 200000,
  "validation_users": 50000,
  "R@5": 0.2579,
  "R@50": 0.53418,
  "MRR@5": 0.150483,
  "MRR@50": 0.17354879384323615
}
